In [2]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from LangChain.app.config import config
from IPython.display import Image

# 初始化 LLM
llm = ChatOpenAI(
    model=config.CONFIG_DEEPSEEK["model"],
    openai_api_key=config.CONFIG_DEEPSEEK["api_key"],
    openai_api_base=config.CONFIG_DEEPSEEK["base_url"],
    temperature=0.7
)


SYSTEM_PROMPT = """你是一个友善、专业的 AI 助手。
你的回答应该：
- 简洁清晰
- 使用中文回复
- 在不确定时主动询问用户
"""

def chatbot_node(state: MessagesState) -> dict:
    """核心对话节点"""
    system = SystemMessage(content=SYSTEM_PROMPT)
    messages = [system] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

# 多轮对话函数
def chat(conversation_history: list, user_input: str) -> tuple[str, list]:
    """处理单轮对话，返回 AI 响应和更新后的历史"""
    conversation_history.append(HumanMessage(content=user_input))
    result = graph.invoke({"messages": conversation_history})
    conversation_history = result["messages"]
    ai_response = conversation_history[-1].content
    return ai_response, conversation_history

# 多轮对话示例
history = []
while True:
    user_input = input("你: ")
    if user_input.lower() in ["退出", "exit", "quit"]:
        print("再见！")
        break
    
    response, history = chat(history, user_input)
    print(f"助手: {response}\n")

# 编译图
graph = builder.compile()

Image(graph.get_graph().draw_mermaid_png())

# 或者打印 Mermaid 格式
print(graph.get_graph().draw_mermaid())

助手: 你好，WLL！很高兴认识你。有什么我可以帮你的吗？

助手: 太好了！Python 是一门非常适合入门的编程语言，语法简洁、应用广泛（Web开发、数据分析、人工智能等）。

我们可以从这几个步骤开始：
1. **安装环境**：推荐安装 Python 最新版 + VS Code（或 PyCharm）。
2. **基础语法**：变量、数据类型、条件判断、循环、函数。
3. **动手练习**：写一些简单的小程序，比如计算器、猜数字游戏。
4. **进阶方向**：根据兴趣选择 Web（Flask/Django）、数据分析（Pandas）或自动化脚本。

你现在是零基础吗？有没有具体想用 Python 实现的目标？我可以根据你的情况推荐更针对性的学习路径。

助手: 当然记得，你叫 WLL。😊

再见！
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

